In [1]:
import operator
from math import log, sqrt

def createDataSet():  # 生成8个样本，此处的labels不是标签，是属性。dataSet的最后一列才是对应的标签是男或者女.数据集是一个多维列表
    dataSet = [[1, '长', '粗', '男'],
               [2, '短', '粗', '男'],
               [3, '短', '粗', '男'],
               [4, '长', '细', '女'],
               [5, '短', '细', '女'],
               [6, '短', '粗', '女'],
               [7, '长', '粗', '女'],
               [8, '长', '粗', '女']]
    labels = ['序号', '头发', '声音']  # three features
    return dataSet, labels


#### 辅助函数，统计样本中不同类别的数目

In [2]:
def classCount(dataSet): #最后得到一个字典{'男'：3，’女':5}
    labelCount = {}
    for one in dataSet:
        if one[-1] not in labelCount.keys():
            labelCount[one[-1]] = 0
        labelCount[one[-1]] += 1
    return labelCount


#### 计算信息熵

In [3]:
def calcShannonEntropy(dataSet):#直接计算数据集的信息熵
    labelCount = classCount(dataSet)
    numEntries = len(dataSet)#数据总长度：8
    Entropy = 0.0
    for i in labelCount:
        prob = float(labelCount[i]) / numEntries
        Entropy -= prob * log(prob, 2)
    return Entropy


### 根据属性对数据集进行分割

#####  辅助函数，找出数据集中比例占多数的类别

In [5]:
def majorityClass(dataSet):#找出数据集中比例占多数的性别
    labelCount = classCount(dataSet)
    sortedLabelCount = sorted(labelCount.items(), key=operator.itemgetter(1), reverse=True)
    return sortedLabelCount[0][0]


##### 根据离散属性进行分割

In [6]:
def splitDataSet(dataSet, i, value):#取出并返回数据集第i个维度上值为value的子集，i为属性。并且返回的数据去除了第i维
    subDataSet = [] 
    for one in dataSet:
        if one[i] == value:
            reduceData = one[:i]
            reduceData.extend(one[i + 1:])#这两行操作的作用是去除第i维数据
            subDataSet.append(reduceData)
    return subDataSet


##### 根据连续属性进行分割

In [7]:
def splitContinuousDataSet(dataSet, i, value,
                           direction):  # split the data according the value, i was axis ,direction 0 is >= and direction 1 is <=
    subDataSet = []
    for one in dataSet:
        if direction == 0:
            if one[i] > value:#direction为0，是取大于的值，direction为1，则是取负方向的值
                reduceData = one[:i]
                reduceData.extend(one[i + 1:])
                subDataSet.append(reduceData)
        if direction == 1:
            if one[i] <= value:
                reduceData = one[:i]
                reduceData.extend(one[i + 1:])
                subDataSet.append(reduceData)
    return subDataSet


##### 选择最优属性

In [8]:
#注意，这里是直接选择最优增益率的属性，没有使用上文中所说的启发式搜索，即先从划分属性中找出信息增益高于平均水平的属性，再从中选择增益率最高的。
def chooseBestFeat(dataSet, labels):
    baseEntropy = calcShannonEntropy(dataSet) #先计算数据集总体的信息熵
    bestFeat = 0 #先假定bestFeat为第0维的属性
    baseGainRatio = -1
    numFeats = len(dataSet[0]) - 1  # the number of features 特征的个数
    bestSplitDic = {}
    i = 0
    print('dataSet[0]:' + str(dataSet[0]))
    for i in range(numFeats):
        featVals = [example[i] for example in dataSet]  # 将dataset中的第i列数据取出
        # print('chooseBestFeat:'+str(i))
        if type(featVals[0]).__name__ == 'float' or type(featVals[0]).__name__ == 'int':# 判断属性是否为连续属性。对连续值进行划分选择，else则是对离散值进行划分选择
            j = 0
            sortedFeatVals = sorted(featVals)#对选中的这列属性值进行升序排列
            splitList = [] #储存二分法的连续属性的节点(这里为7个值)
            for j in range(len(featVals) - 1):
                splitList.append((sortedFeatVals[j] + sortedFeatVals[j + 1]) / 2.0)
            for j in range(len(splitList)):
                newEntropy = 0.0
                gainRatio = 0.0
                splitInfo = 0.0
                value = splitList[j] #对二分节点依次进行遍历
                subDataSet0 = splitContinuousDataSet(dataSet, i, value, 0)#大于value的数据集
                subDataSet1 = splitContinuousDataSet(dataSet, i, value, 1)#小于value的数据集
                prob0 = float(len(subDataSet0)) / len(dataSet)
                newEntropy -= prob0 * calcShannonEntropy(subDataSet0)
                prob1 = float(len(subDataSet1)) / len(dataSet)
                newEntropy -= prob1 * calcShannonEntropy(subDataSet1)
                splitInfo -= prob0 * log(prob0, 2)
                splitInfo -= prob1 * log(prob1, 2)
                gainRatio = float(baseEntropy - newEntropy) / splitInfo#将信息增益转化为增益率，之前也说过这里是C4.5决策树算法。
                print('IVa ' + str(j) + ':' + str(splitInfo))
                if gainRatio > baseGainRatio:
                    baseGainRatio = gainRatio
                    bestSplit = j
                    bestFeat = i
            bestSplitDic[labels[i]] = splitList[bestSplit]
        else:#对离散属性值进行划分
            uniqueFeatVals = set(featVals)
            GainRatio = 0.0
            splitInfo = 0.0
            newEntropy = 0.0
            for value in uniqueFeatVals:
                subDataSet = splitDataSet(dataSet, i, value)
                prob = float(len(subDataSet)) / len(dataSet)
                splitInfo -= prob * log(prob, 2)
                newEntropy -= prob * calcShannonEntropy(subDataSet)
            gainRatio = float(baseEntropy - newEntropy) / splitInfo
            if gainRatio > baseGainRatio:
                bestFeat = i
                baseGainRatio = gainRatio
    if type(dataSet[0][bestFeat]).__name__ == 'float' or type(dataSet[0][bestFeat]).__name__ == 'int':
        bestFeatValue = bestSplitDic[labels[bestFeat]]
        ##bestFeatValue=labels[bestFeat]+'<='+str(bestSplitValue)
    if type(dataSet[0][bestFeat]).__name__ == 'str':
        bestFeatValue = labels[bestFeat]
    return bestFeat, bestFeatValue


##### 创建决策树

In [9]:
def createTree(dataSet, labels):
    classList = [example[-1] for example in dataSet] #直接取了数据集中的最后一列作为标签
    if len(set(classList)) == 1:#这四行为递归终止条件。若类别中只剩下一项时，停止递归；
        return classList[0][0]
    if len(dataSet[0]) == 1:#若数据集中只剩下一项属性值时，直接根据按比例生成树
        return majorityClass(dataSet)
    Entropy = calcShannonEntropy(dataSet)#计算数据集的信息熵
    bestFeat, bestFeatLabel = chooseBestFeat(dataSet, labels)#选择当前数据集中的最优属性
    print('bestFeat:' + str(bestFeat) + '--' + str(labels[bestFeat]) + ', bestFeatLabel:' + str(bestFeatLabel))
    myTree = {labels[bestFeat]: {}}#建立一个集合用来存放树结构
    subLabels = labels[:bestFeat]
    subLabels.extend(labels[bestFeat + 1:])#这两行的作用是将属性labels中除最优属性外的其他属性拿出来
    print('subLabels:' + str(subLabels))
    if type(dataSet[0][bestFeat]).__name__ == 'str':
        featVals = [example[bestFeat] for example in dataSet]
        uniqueVals = set(featVals)
        print('uniqueVals:' + str(uniqueVals))
        for value in uniqueVals:#递归调用
            reduceDataSet = splitDataSet(dataSet, bestFeat, value)
            print('reduceDataSet:' + str(reduceDataSet))
            myTree[labels[bestFeat]][value] = createTree(reduceDataSet, subLabels)
    if type(dataSet[0][bestFeat]).__name__ == 'int' or type(dataSet[0][bestFeat]).__name__ == 'float':
        value = bestFeatLabel
        #将数据集根据最优属性值进行划分，划分成两个子集
        greaterDataSet = splitContinuousDataSet(dataSet, bestFeat, value, 0)
        smallerDataSet = splitContinuousDataSet(dataSet, bestFeat, value, 1)
        print('greaterDataset:' + str(greaterDataSet))
        print('smallerDataSet:' + str(smallerDataSet))
        print('== ' * len(dataSet[0]))
        myTree[labels[bestFeat]]['>' + str(value)] = createTree(greaterDataSet, subLabels)
        print(myTree)
        print('== ' * len(dataSet[0]))
        myTree[labels[bestFeat]]['<=' + str(value)] = createTree(smallerDataSet, subLabels)
    return myTree


##### 将决策树打印出来

In [10]:
if __name__ == '__main__':
    dataSet, labels = createDataSet()
    tree=createTree(dataSet, labels)
    print(tree)


dataSet[0]:[1, '长', '粗', '男']
IVa 0:0.5435644431995964
IVa 1:0.8112781244591328
IVa 2:0.9544340029249649
IVa 3:1.0
IVa 4:0.9544340029249649
IVa 5:0.8112781244591328
IVa 6:0.5435644431995964
bestFeat:0--序号, bestFeatLabel:7.5
subLabels:['头发', '声音']
greaterDataset:[['长', '粗', '女']]
smallerDataSet:[['长', '粗', '男'], ['短', '粗', '男'], ['短', '粗', '男'], ['长', '细', '女'], ['短', '细', '女'], ['短', '粗', '女'], ['长', '粗', '女']]
== == == == 
{'序号': {'>7.5': '女'}}
== == == == 
dataSet[0]:['长', '粗', '男']
bestFeat:0--头发, bestFeatLabel:头发
subLabels:['声音']
uniqueVals:{'短', '长'}
reduceDataSet:[['粗', '男'], ['粗', '男'], ['细', '女'], ['粗', '女']]
dataSet[0]:['粗', '男']
bestFeat:0--声音, bestFeatLabel:声音
subLabels:[]
uniqueVals:{'细', '粗'}
reduceDataSet:[['女']]
reduceDataSet:[['男'], ['男'], ['女']]
reduceDataSet:[['粗', '男'], ['细', '女'], ['粗', '女']]
dataSet[0]:['粗', '男']
bestFeat:0--声音, bestFeatLabel:声音
subLabels:[]
uniqueVals:{'细', '粗'}
reduceDataSet:[['女']]
reduceDataSet:[['男'], ['女']]
{'序号': {'>7.5': '女', '<=7.5': {'头发'